This notebook follows the place-name spike workflow in [`spikes_edward_scotland.ipynb`](./spikes_edward_scotland.ipynb), looking for mentions of Normandy during Henry I's reign and examining people mentioned near a spike.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import xml.etree.ElementTree as ET
import re
import unicodedata
import textwrap

# The text and searched names go through the same spelling normalization.
def norm(word):
    """Normalize common early-modern spelling variants in a word."""
    word = word.replace("ã", "an").replace("õ", "on").replace("ẽ", "en").replace("ũ", "un")
    word = unicodedata.normalize("NFKD", word).encode("ascii", "ignore").decode()
    key = word.lower().replace("vv", "w").replace("j", "i").replace("v", "u").replace("y", "i")
    key = re.sub(r"au(?=n)", "a", key)
    key = re.sub(r"(.)\1", r"\1", key)
    key = re.sub(r"e$", "", key)
    return key.capitalize() if word[:1].isupper() else key

def norm_text(text):
    """Normalize each word while retaining digits and punctuation."""
    return " ".join(re.sub(r"[^\W\d_]+", lambda match: norm(match.group()), text).split())

def name_pattern(spellings):
    """Build whole-word patterns from pipe-separated spelling variants."""
    keys = sorted({norm_text(s) for s in spellings.split("|") if s.strip()}, key=len, reverse=True)
    return re.compile(r"(?<!\w)(?:" + "|".join(
        f"(?i:{re.escape(k)})" if " " in k else k[0].upper() + f"(?i:{re.escape(k[1:])})" for k in keys
    ) + r")(?!\w)")

def word_index(text, char_pos):
    """Return the 0-based word position of a character offset."""
    return len(text[:char_pos].split())

In [ ]:
repo_root = Path.cwd().resolve()
if not (repo_root / "1577_files").is_dir():
    repo_root = repo_root.parent

henry_path = repo_root / "1577_files" / "regnal_files" / "03_HenryFirst.xml"
TEI = "{http://www.tei-c.org/ns/1.0}"
root = ET.parse(henry_path).getroot()
henry_entry = root.find(f".//{TEI}div[@type='reign'][@n='Henry_I']")

if henry_entry is None:
    raise ValueError("Could not find Henry I's reign section.")

henry_xml_words = norm_text(" ".join(henry_entry.itertext()))
henry_words = henry_xml_words.split()
print(textwrap.fill(henry_xml_words[:1000], width=150))

In [ ]:
# Normandy appears in the source with spellings such as Normandie, Normandy,
# and Normandye; normalization maps these to the same whole-word form.
normandy_pattern = name_pattern("Normandie|Normandy|Normandye")
word_positions = [
    word_index(henry_xml_words, match.start())
    for match in normandy_pattern.finditer(henry_xml_words)
]
print(f"Found {len(word_positions)} mentions of Normandy.")

# Count mentions in consecutive 500-word sections.
window_size = 500
number_of_windows = (len(henry_words) + window_size - 1) // window_size
normandy_counts = (
    pd.Series(word_positions)
    .floordiv(window_size)
    .value_counts()
    .sort_index()
    .reindex(range(number_of_windows), fill_value=0)
)

# Extract marginal years (1100-1135) and map each to its position in the text.
year_pattern = re.compile(r"(?<!\d)11(?:0[0-9]|[12][0-9]|3[0-5])(?!\d)")
year_markers = []
search_from = 0
for note in henry_entry.findall(f".//{TEI}note[@place='marg']"):
    note_text = norm_text(" ".join(note.itertext()))
    matches = list(year_pattern.finditer(note_text))
    if not matches:
        continue
    note_offset = henry_xml_words.find(note_text, search_from)
    if note_offset < 0:
        raise ValueError(f"Could not locate marginal note in extracted text: {note_text!r}")
    for match in matches:
        year_markers.append((int(match.group()), word_index(henry_xml_words, note_offset + match.start())))
    search_from = note_offset + len(note_text)

display(normandy_counts.sort_values(ascending=False).head(10))

plt.figure(figsize=(16, 5))
plt.bar(normandy_counts.index * window_size, normandy_counts.values, width=window_size * 0.9, align="edge")
ax = plt.gca()
max_count = normandy_counts.max()
for i, (year, word_position) in enumerate(year_markers):
    ax.axvline(word_position, color="darkred", alpha=0.25, linewidth=0.8)
    ax.text(word_position, max_count + 1 + (i % 4) * 2.5, str(year), rotation=90,
            ha="center", va="bottom", color="darkred", fontsize=8)
ax.set_ylim(0, max_count + 13)
plt.xlabel("Word in Henry I's text")
plt.ylabel("Mentions per 500-word section")
plt.title("Normandy Mentions Across Henry I's Reign")
plt.xticks(ticks=range(0, len(henry_words) + 1, 2000), rotation=45)
plt.tight_layout()
plt.show()

The strongest cluster is around words 5,500–7,500. The following cells compare the people named in that passage and plot their mentions against the Normandy spike.

In [ ]:
people_df = pd.read_csv(repo_root / "name_extraction" / "entity_names" / "holinshed_people.csv")

# Merge rows with equivalent normalized names. For people, spelling_variants
# contains short forms that can match common words too broadly, so use names only.
people_df = people_df.groupby(people_df["name"].map(norm_text).str.lower(), sort=False).agg(
    name=("name", "first"), spellings=("name", "|".join)
)
patterns = {row["name"]: name_pattern(row["spellings"]) for _, row in people_df.iterrows()}

slice_start = 5500
slice_end = 7500
spike_text = " ".join(henry_words[slice_start:slice_end])
people_df["count"] = people_df["name"].map(
    lambda name: len(patterns[name].findall(spike_text))
)
people_in_slice = (
    people_df[people_df["count"] > 0][["name", "count"]]
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)
display(people_in_slice.head(10))

In [ ]:
top_people = people_in_slice.head(8)["name"].tolist()
person_positions = {
    name: [slice_start + word_index(spike_text, match.start()) for match in patterns[name].finditer(spike_text)]
    for name in top_people
}

fig, (ax_bar, ax_people) = plt.subplots(
    2, 1, figsize=(16, 8), sharex=True, gridspec_kw={"height_ratios": [1, 1.2]}
)
ax_bar.bar(normandy_counts.index * window_size, normandy_counts.values,
           width=window_size * 0.9, align="edge")
ax_bar.set_ylabel("Mentions per 500-word section")
ax_bar.set_title("Normandy Mentions Across Henry I's Reign")

colors = plt.cm.tab10.colors
for i, (name, positions) in enumerate(person_positions.items()):
    ax_people.eventplot(positions, lineoffsets=i, linelengths=0.8, linewidths=2,
                        colors=colors[i % len(colors)])
ax_people.set_yticks(range(len(person_positions)))
ax_people.set_yticklabels(list(person_positions.keys()))
if person_positions:
    ax_people.set_ylim(-0.5, len(person_positions) - 0.5)
    ax_people.invert_yaxis()
ax_people.set_xlabel("Word in Henry I's text")
ax_people.set_xticks(range(slice_start, slice_end + 1, 500))
ax_people.tick_params(axis="x", rotation=45)
ax_people.set_xlim(slice_start, slice_end)
fig.tight_layout()
plt.show()

In [ ]:
# Inspect a 250-word passage near the early Normandy spike.
section = " ".join(henry_words[5750:6000])
print(textwrap.fill(section, width=150))